# Micronic RD235 Code Reader

The Micronic RD235 Code Reader scans the side barcode of an 8x12 tube rack, acquires an image of
the rack, and decodes the 96 tube DataMatrix codes locally. It does not require Micronic Code
Reader or IO Monitor.

| Property | Value |
|---|---|
| Model | RD235 |
| Rack-ID communication | Serial trigger and response |
| Serial settings | 9600 baud, 7 data bits, even parity, 1 stop bit |
| Rack image | TWAIN helper on Windows, SANE `scanimage` on Linux, or a custom `Scanner` |
| Rack layout | 8 rows by 12 columns |
| Tube barcode | Ten-digit DataMatrix |
| Rack barcode | Code 128 |

[OEM link](https://www.micronic.com/products/code-reader/)

```{device-card} micronic-code-reader
```

## How it talks

The side reader uses `pylabrobot.io.Serial`. A rack-ID scan clears pending input, sends
`<t>` followed by CRLF, reads through the response terminator, and returns the first sequence
of at least six digits. It returns `NOREAD` when the response contains no rack identifier.

Rack-image acquisition runs through PyLabRobot's command-line transport. `TwainScanner` calls
an operator-installed TWAIN helper; `SaneScanner` calls `scanimage`. PyLabRobot then performs
grid fitting and DataMatrix decoding locally.

## Physical setup

1. Connect the rack scanner to the computer and install its operating-system driver.
2. Connect the side rack-ID reader and identify its serial port.
3. On Windows, install the local TWAIN helper and note its path and TWAIN source name. On
   Linux, install SANE and confirm that `scanimage --list-devices` lists the scanner.
4. Install the serial and decoding dependencies: `pylabrobot[serial]`, `pillow`,
   `opencv-python-headless`, `numpy`, and `zxing-cpp`.
5. Place an 8x12 Micronic tube rack in the scanner with the orientation expected by the
   instrument.

## Connect

Choose the scanner implementation for the host. This example uses a Windows TWAIN helper. On
Linux, replace it with `SaneScanner(sane_device="<device from scanimage --list-devices>")`.
`setup()` resolves the scanner executable, creates the image directory, and opens the side
reader's serial port.

In [ ]:
from pylabrobot.micronic import MicronicRD235, TwainScanner

reader = MicronicRD235(
  scanner=TwainScanner(
    twain_scanner_path=r"C:\Tools\twain_scan.exe",
    twain_source="AVA6PlusG",
  ),
  serial_port="COM4",
  image_dir=r"C:\ProgramData\PyLabRobot\micronic-images",
)
await reader.setup()

## Read the rack ID

`scan_rack_id()` triggers only the side barcode reader. It returns the decoded identifier or
`NOREAD`.

In [ ]:
rack_id = await reader.scan_rack_id(timeout=5.0)
print(rack_id)

## Represent the rack

`scan_rack()` accepts a tube rack in a resource store with exactly 8 rows and 12 columns. Define
the rack layout before scanning it.

In [ ]:
from pylabrobot.resources import Coordinate
from pylabrobot.resources.store import ResourceStore, build
from pylabrobot.resources.store.v1 import types_pb2 as pb

store = ResourceStore.in_memory()
holder = build.resource(
  "micronic_96_tube_rack_resourceholder_{identifier}",
  Coordinate(9.0, 9.0, 20.0),
  kind=pb.RESOURCE_KIND_RESOURCE_HOLDER,
  holder=pb.HolderSpec(capacity=1),
)
rack = await store.create(
  build.resource(
    "micronic_96_tube_rack",
    Coordinate(85.0, 127.0, 20.0),
    kind=pb.RESOURCE_KIND_TUBE_RACK,
    itemized=pb.ItemizedSpec(
      num_items_x=12,
      num_items_y=8,
      grid=pb.ItemGrid(item=holder, item_dx=9.0, item_dy=9.0),
    ),
  ),
  name="micronic_96_tube_rack",
)

## Scan the rack

`scan_rack()` reads the side rack barcode, acquires the rack image, and returns one entry for
each tube position. A position reports `NOREAD` when no tube barcode was decoded.

In [ ]:
result = await reader.scan_rack(rack=rack, timeout=90.0)
print(result.rack_id)
print(len([entry for entry in result.entries if entry.status == "OK"]))

## Record the barcodes once

The reader returns the codes it reads; it never writes them into the resource store. A rack and
tubes whose barcodes the store does not record yet are labelled once with `initialize_barcodes`.
It refuses to record a barcode over a different one.

In [ ]:
from pylabrobot.micronic import rack_scan_reads
from pylabrobot.resources.store.barcodes import barcodes_by_place, initialize_barcodes

await initialize_barcodes(barcodes_by_place(rack_scan_reads(rack, result)))

## Check a scan against the store

With `check=True`, `scan_rack()` compares what it reads with the barcodes the store records, and
raises `BarcodeMismatchError` when a tube was moved or swapped, a recorded tube is missing, or a
tube is found where the store holds none.

In [ ]:
result = await reader.scan_rack(rack=rack, timeout=90.0, check=True)

## Disconnect

`stop()` closes the side reader's serial connection.

In [ ]:
await reader.stop()